## Úvod 

Táto lekcia pokryje: 
- Čo je volanie funkcie a jeho použitia 
- Ako vytvoriť volanie funkcie pomocou Azure OpenAI 
- Ako integrovať volanie funkcie do aplikácie 

## Ciele učenia 

Po dokončení tejto lekcie budete vedieť a rozumieť: 

- Účelu použitia volania funkcie 
- Nastaveniu volania funkcie pomocou Azure Open AI Service 
- Navrhnúť efektívne volania funkcií pre prípad použitia vašej aplikácie 


## Pochopenie volania funkcií 

Pre túto lekciu chceme vytvoriť funkciu pre náš vzdelávací startup, ktorá umožní používateľom vyhľadávať technické kurzy pomocou chatbota. Budeme odporúčať kurzy, ktoré zodpovedajú ich úrovni zručností, aktuálnej pozícii a záujmu o technológie. 

Na dokončenie použijeme kombináciu: 
 - `Azure Open AI` na vytvorenie chatovacieho zážitku pre používateľa
 - `Microsoft Learn Catalog API` na pomoc používateľom vyhľadávať kurzy podľa požiadavky používateľa 
 - `Function Calling` na prijatie dotazu používateľa a jeho odoslanie do funkcie pre vykonanie API dopytu. 

Na začiatok si pozrime, prečo by sme chceli vôbec používať volanie funkcií: 


### Prečo volanie funkcií 

Ak ste absolvovali niektorú z predchádzajúcich lekcií v tomto kurze, pravdepodobne rozumiete sile používania veľkých jazykových modelov (LLM). Dúfajme, že vidíte aj niektoré ich obmedzenia. 

Volanie funkcií je funkcia služby Azure Open AI, ktorá prekonáva nasledujúce obmedzenia: 
1) Konzistentný formát odpovedí 
2) Schopnosť použiť dáta z iných zdrojov aplikácie v kontexte chatu 

Pred volaním funkcií boli odpovede z LLM nestruktúrované a nekonzistentné. Vývojári museli písať zložitý validačný kód, aby zabezpečili, že dokážu spracovať každú variáciu odpovede. 

Používatelia nedokázali získať odpovede ako „Aké je aktuálne počasie v Štokholme?“. Je to preto, že modely boli obmedzené na čas, kedy boli dáta trénované. 

Pozrime sa na príklad nižšie, ktorý ilustruje tento problém: 

Povedzme, že chceme vytvoriť databázu údajov o študentoch, aby sme im mohli navrhnúť ten správny kurz. Nižšie máme dva popisy študentov, ktoré sú veľmi podobné z hľadiska obsahovaných dát. 


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Chceme to poslať do LLM, aby rozanalyzoval údaje. To môže byť neskôr použité v našej aplikácii na odoslanie do API alebo uloženie do databázy. 

Vytvorme dva identické prompti, ktorými LLM nasmerujeme, aké informácie nás zaujímajú: 


Chceme to poslať LLM, aby analyzoval časti, ktoré sú dôležité pre náš produkt. Takže môžeme vytvoriť dva zhodné prompti na inštruovanie LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Po vytvorení týchto dvoch promptov ich odošleme do LLM pomocou `client.responses.create`.  Prompt uložíme do premennej `input` a rolu priradíme k `user`. Toto má napodobniť správu od používateľa, ktorá sa zapisuje do chatbotu. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Teraz môžeme odoslať obe požiadavky do LLM a preskúmať odpoveď, ktorú dostaneme. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Aj keď sú výzvy rovnaké a popisy podobné, môžeme dostať rôzne formáty vlastnosti `Grades`. 

Ak spustíte vyššie uvedenú bunku viackrát, formát môže byť `3.7` alebo `3.7 GPA`. 

Je to preto, že LLM prijíma neštruktúrované dáta vo forme napísanej výzvy a tiež vracia neštruktúrované dáta. Potrebujeme mať štruktúrovaný formát, aby sme vedeli, čo očakávať pri ukladaní alebo používaní týchto dát.

Použitím funkčného volania môžeme zabezpečiť, že dostaneme späť štruktúrované dáta. Pri používaní funkčného volania LLM v skutočnosti nevolá alebo nespúšťa žiadne funkcie. Namiesto toho vytvoríme štruktúru, ktorej má LLM pri svojich odpovediach dodržiavať. Potom použijeme tieto štruktúrované odpovede, aby sme vedeli, ktorú funkciu spustiť v našich aplikáciách.  
 


![Diagram priebehu volania funkcie](../../../../translated_images/sk/Function-Flow.083875364af4f4bb.webp)


Potom môžeme vziať to, čo funkcia vráti, a odoslať to späť do LLM. LLM potom odpovie pomocou prirodzeného jazyka, aby zodpovedal používateľovu otázku. 


### Prípady použitia volaní funkcií 

**Volanie externých nástrojov** 
Chatboty sú skvelé pri poskytovaní odpovedí na otázky používateľov. Vďaka volaniu funkcií môžu chatboty využívať správy od používateľov na dokončenie určitých úloh. Napríklad študent môže požiadať chatbota, aby „Poslal e-mail môjmu inštruktorovi s informáciou, že potrebujem viac pomoci s touto témou“. Toto môže viesť k volaniu funkcie `send_email(to: string, body: string)`


**Vytváranie API alebo databázových dotazov**
Používatelia môžu nájsť informácie pomocou prirodzeného jazyka, ktorý sa následne premení na formátovaný dotaz alebo API požiadavku. Príkladom je učiteľ, ktorý požaduje „Ktorí študenti dokončili poslednú úlohu“, čo môže vyvolať volanie funkcie s názvom `get_completed(student_name: string, assignment: int, current_status: string)`


**Vytváranie štruktúrovaných údajov**
Používatelia môžu vziať blok textu alebo CSV a využiť LLM na extrakciu dôležitých informácií z neho. Napríklad študent môže previesť článok z Wikipédie o mierových dohodách na vytvorenie AI flash kariet. Toto je možné vykonať pomocou funkcie nazvanej `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Vytvorenie vášho prvého volania funkcie 

Proces vytvárania volania funkcie zahŕňa 3 hlavné kroky: 
1. Volanie Chat Completions API so zoznamom vašich funkcií a správou od používateľa 
2. Prečítať odpoveď modelu, aby ste vykonali akciu, napríklad spustili funkciu alebo API volanie 
3. Vykonať ďalšie volanie Chat Completions API s odpoveďou z vašej funkcie, aby ste použili tieto informácie na vytvorenie odpovede pre používateľa. 


![Tok volania funkcie](../../../../translated_images/sk/LLM-Flow.3285ed8caf4796d7.webp)


### Prvky volania funkcie 

#### Užívateľský vstup 

Prvým krokom je vytvoriť správu od užívateľa. Tá môže byť dynamicky priradená z hodnoty textového vstupu alebo môžete hodnotu priradiť tu. Ak pracujete s API Chat Completions prvýkrát, potrebujeme definovať `role` a `content` správy. 

`role` môže byť `system` (vytváranie pravidiel), `assistant` (model) alebo `user` (koncový užívateľ). Pre volanie funkcie priradíme túto hodnotu ako `user` a príklad otázky. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Vytváranie funkcií. 

Ďalej si definujeme funkciu a parametre tejto funkcie. Použijeme len jednu funkciu s názvom `search_courses`, ale môžete vytvoriť viacero funkcií.

**Dôležité** : Funkcie sú súčasťou systémovej správy pre LLM a budú zahrnuté do množstva dostupných tokenov, ktoré máte k dispozícii.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definície** 

`name` -  Názov funkcie, ktorú chceme mať zavolanú. 

`description` - Toto je popis toho, ako funkcia funguje. Tu je dôležité byť konkrétny a jasný 

`parameters` - Zoznam hodnôt a formát, ktorý chcete, aby model vyprodukoval vo svojej odpovedi 


`type` -  Typ údajov, v ktorom sa budú vlastnosti ukladať 

`properties` - Zoznam konkrétnych hodnôt, ktoré model použije vo svojej odpovedi 


`name` - názov vlastnosti, ktorú model použije vo svojej formátovanej odpovedi 

`type` - Typ údajov tejto vlastnosti 

`description` - Popis konkrétnej vlastnosti 


**Nepovinné**

`required` - vyžadovaná vlastnosť pre dokončenie volania funkcie 


### Volanie funkcie 
Po definovaní funkcie ju teraz potrebujeme zahrnúť do volania API pre Chat Completion. Robíme to pridaním `functions` do požiadavky. V tomto prípade `functions=functions`. 

Je tu aj možnosť nastaviť `function_call` na `auto`. To znamená, že necháme LLM rozhodnúť, ktorá funkcia by sa mala vyvolať na základe správy používateľa, namiesto toho, aby sme to priraďovali sami.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Teraz sa pozrime na odpoveď a uvidíme, ako je naformátovaná: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Vidíte, že je zavolaná funkcia podľa názvu a na základe správy používateľa LLM dokázal nájsť dáta, ktoré zodpovedajú argumentom funkcie. 


## 3. Integrácia volaní funkcií do aplikácie. 


Po otestovaní formátovanej odpovede z LLM ju môžeme integrovať do aplikácie. 

### Správa toku 

Na integráciu do našej aplikácie vykonajme nasledujúce kroky: 

Najprv vykonajme volanie na služby Open AI a uložme správu do premennej nazvanej `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Teraz definujeme funkciu, ktorá zavolá Microsoft Learn API na získanie zoznamu kurzov: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Ako najlepšiu prax potom zistíme, či model chce zavolať funkciu. Následne vytvoríme jednu z dostupných funkcií a priradíme ju k volanej funkcii. 
Potom vezmeme argumenty funkcie a namapujeme ich na argumenty z LLM.

Nakoniec pripojíme správu volania funkcie a hodnoty, ktoré boli vrátené správou `search_courses`. To poskytuje LLM všetky informácie, ktoré potrebuje
na odpoveď používateľovi pomocou prirodzeného jazyka. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Teraz pošleme aktualizovanú správu do LLM, aby sme mohli dostať odpoveď v prirodzenom jazyku namiesto odpovede formátovanej v API JSON. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Kódová výzva

Skvelá práca! Na pokračovanie vo vašom učení sa Azure Open AI Function Calling môžete vytvoriť: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst
 - Viac parametrov funkcie, ktoré môžu pomôcť študentom nájsť viac kurzov. Dostupné parametre API nájdete tu:
 - Vytvorte ďalší volanie funkcie, ktoré zoberie viac informácií od študenta, napríklad ich rodný jazyk
 - Vytvorte spracovanie chýb, keď volanie funkcie a/alebo volanie API nevráti žiadne vhodné kurzy


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
